In [2]:
import json
from openai import OpenAI
from pydantic import BaseModel,Field
from typing import Optional
import openai

In [3]:
client = OpenAI(
    api_key="token-vulcan",  # 输入你的 API Key
    base_url="http://219.147.99.170:40019/v1"
)




def qa_base(messages):
    completion = client.chat.completions.create(
        model="Qwen2-5-14B",
        messages=messages,
        logprobs=False,

        # stream=True  # 开启流式输出,
    )
    return completion.choices[0].message.content
    # if compile.status_code == 200:
    #     result = completion[0].choices[0].delta.content
    #     return result
    # else:
    #     print(f"请求失败，状态码: {completion.status_code}")
    #     return ""
    

In [4]:
user_input = "中国的首都是哪里"
input = [{"role": "user", "content": user_input},]
qa_base(input)

'中国的首都是北京。'

In [5]:
with open("./life_hierarchy.json","r")as f:
    life_data = json.load(f)

# 构建第一层tot 问题

In [18]:
time_of_illness = []
question_map_level_1 = {}
for conclusion in life_data["Life"]:
    name_1 = conclusion["label"]
    print(name_1)
    question_map_level_1[name_1] = conclusion
    time_of_illness.append(name_1)
question_level_1 = f"""
    请判断该病人的病程是如下的那种情况，病程情况：{time_of_illness[:-1]},
    注意返回结果只能是病程情况中的一种，不要有其他介绍说明
    """
print(question_level_1)

病程小于6个月
病程为6个月－1年
病程为1年－15年
病程>15年
存在并发症和/或其他风险因素

    请判断该病人的病程是如下的那种情况，病程情况：['病程小于6个月', '病程为6个月－1年', '病程为1年－15年', '病程>15年'],
    注意返回结果只能是病程情况中的一种，不要有其他介绍说明
    


# 构建第二层tot 问题
## 获得上一次答案后，可以根据答案map到相应的分支
### 到下层分支时有两种情况，第一种就是到叶子节点了(即，value!=""),第二种就是未到叶子节点(children!=[])
### 到达叶子结点有两种情况，一种是得到了评点结论，如”延期“、”拒保“等，该种情况下直接返回结论；另外一种是得到了评点值，如+200等，这种情况下将继续走到"存在并发症和/或其他风险因素"分支接续判断

In [32]:
answer = "病程小于6个月"
answer = "病程为6个月－1年"
value = question_map_level_1[answer]["value"]
if value!="":
    print(f"直接得出结论：{value}")
else:
    tree_branch = question_map_level_1[answer]
    print(tree_branch)
    children_nodes = tree_branch["children"]
    print(children_nodes)
    node_questions = []
    for node in children_nodes:
        node_questions.append(node["label"])
    print(node_questions)
    question_level_2 = f"当病人存在{answer}时，请判断是否有如下情况，情况列表：{node_questions},注意返回结果只能是情况列表中的一种，不要有其他介绍说明"

# print("==============\n")
# for key,value in question_map_level_1.items():
#     if key == "病程小于6个月" or key == "存在并发症和/或其他风险因素":
#         continue
#     print("key:",key)
#     concl_value  = value["value"]
#     if concl_value!="":
#         print(concl_value)


{'label': '病程为6个月－1年', 'value': '', 'children': [{'label': '<15岁', 'value': '延期', 'children': []}, {'label': '15-19岁', 'value': '', 'children': [{'label': 'HbA1c < 7%', 'value': '+200', 'children': []}, {'label': 'HbA1c ≥ 7%', 'value': '拒保', 'children': []}]}, {'label': '20-39岁', 'value': '', 'children': [{'label': 'HbA1c < 7%', 'value': '+150', 'children': []}, {'label': 'HbA1c 7.1-8%', 'value': '+200', 'children': []}, {'label': 'HbA1c 8.1-10%', 'value': '+250', 'children': []}, {'label': 'HbA1c > 10%', 'value': '拒保', 'children': []}]}, {'label': '40-49岁', 'value': '', 'children': [{'label': 'HbA1c < 7%', 'value': '+100', 'children': []}, {'label': 'HbA1c 7.1-8%', 'value': '+125', 'children': []}, {'label': 'HbA1c 8.1-10%', 'value': '+150', 'children': []}, {'label': 'HbA1c > 10%', 'value': '拒保', 'children': []}]}, {'label': '≥50岁', 'value': '', 'children': [{'label': 'HbA1c < 7%', 'value': '+75', 'children': []}, {'label': 'HbA1c 7.1-8%', 'value': '+100', 'children': []}, {'label': 

In [6]:
import re

def is_number(string):
    # Compile the regular expression
    pattern = re.compile(r'^[+-]?(\d+(\.\d*)?|\.\d+)([eE][+-]?\d+)?$')
    # Match the input string
    return bool(pattern.match(string))

# Test cases
test_strings = ["延期","+200"]
results = {s: is_number(s) for s in test_strings}
print(results)


{'延期': False, '+200': True}


# 第一步，针对病程Tree进行多轮问答

In [10]:
input_data = "年龄：71岁\
                临床诊断化验项：_GLU（空腹血糖）:8.82_ALP（碱性磷酸酶）:63.4_GGT（谷氨酰转肽酶）:32.2_TBIL（总胆红素）:5.84_DBIL（直接胆红素）:1.96_\
                AST（谷草转氨酶）:19.70_ALT（谷丙转氨酶）:18.30_TG（甘油三脂）:1.47_BUN（血尿素氮）:5.86_UA（尿酸）:307.50_CR(肌酐):43.60_糖化血红蛋白:9.12_\
                    HBSAG（乙肝表面抗原）:阴性(-)_HDL（高密度脂蛋白）:1.24_LDL（低密度脂蛋白）:2.29_糖尿病"

In [24]:
# input_data = "" #case 疾病信息
historys_list = []
message = {"role": "user", "content": f"你是一个保险公司的专业核保老师，根据提供的病人基本信息回答问题，病人基本信息:{input_data}"}
historys_list.append(message)
life_data_map = {"Life":[concl for concl in life_data["Life"] if concl["label"]!="存在并发症和/或其他风险因素"]}
key = "Life"
life_data_list = life_data_map[key]
question_prompt = """
    请判断该病人是否有如下的情况，情况列表：{node_questions},
    注意返回结果只能是情况列表中的一种，并给出理由，但不要给其他建议说明
    注意返回结果时：把从列表中选择出的值放到最后，用#和前面的分析理由分隔
    结果示例：分析理由#选项值
    """
#如果无法更具给定的数据判断则返回其他

run_flag = True
level_num = 0
while run_flag:
    print("level num:",level_num)
    node_questions = []
    question_map_level = {}
    for conclusion in life_data_list:
        name_1 = conclusion["label"]
        question_map_level[name_1] = conclusion
        node_questions.append(name_1)
    print(node_questions)
    question_prompt_input = question_prompt.format(node_questions=node_questions)
    print(question_prompt_input)
    message = {"role": "user", "content": question_prompt_input}
    historys_list.append(message)
    messages = historys_list
    # print(messages)
    answer = qa_base(messages)
    historys_list.append({"role": "assistant", "content": answer})
    print("answer:",answer)
    # # answer = "病程小于6个月"
    # if level_num==0:
    #     answer = '病程为6个月－1年'
    #     historys_list.append({"role": "assistant", "content": answer})
    # if level_num == 1:
    #     answer = "15-19岁"
    #     historys_list.append({"role": "assistant", "content": answer})
    # if level_num == 2:
    #     answer = "HbA1c < 7%"
    #     historys_list.append({"role": "assistant", "content": answer})
    level_num += 1
    #

    answer_key = answer.split("#")[-1]
    #判断是否能得出最终结论
    other_answer = {"children":[],"value":f"answer值：{answer_key},该值未在给定选项中"}
    # print(answer)
    # print(question_map_level[answer])
    next_level_qa = question_map_level.get(answer_key,other_answer)
    children_value = next_level_qa["children"]
    #children_value==[] 表示到达了叶子结点，否则继续走分支
    if children_value==[]:
        end_result = next_level_qa["value"]
        print("最终结论",end_result)
        break
    else:
        life_data_list = children_value
        print(children_value)

first_end_result = end_result
print("第一步结论：",first_end_result)
print(historys_list)



level num: 0
['病程小于6个月', '病程为6个月－1年', '病程为1年－15年', '病程>15年']

    请判断该病人是否有如下的情况，情况列表：['病程小于6个月', '病程为6个月－1年', '病程为1年－15年', '病程>15年'],
    注意返回结果只能是情况列表中的一种，并给出理由，但不要给其他建议说明
    注意返回结果时：把从列表中选择出的值放到最后，用#和前面的分析理由分隔
    结果示例：分析理由#选项值
    
answer: 根据提供的病人基本信息，该病人年龄为71岁，且临床诊断中明确标示有糖尿病。此外，其糖化血红蛋白（HbA1c）值为9.12%，这个数值显著高于正常范围（通常认为正常值在4%-5.6%之间，但一般不超过6.5%即可诊断为糖尿病），提示患者血糖控制不佳，且其空腹血糖（GLU）为8.82 mmol/L，也高于正常范围（通常在3.9-6.1 mmol/L），进一步证实了其糖尿病状态。虽然没有直接提供糖尿病的病程信息，但基于上述指标，可以合理推测其糖尿病病程可能较长，血糖控制较差。在没有其他信息表明病程较短的情况下，推测其病程超过15年的可能性较大。

分析理由#病程>15年
[{'label': '15-19岁', 'value': '', 'children': [{'label': 'HbA1c < 7%', 'value': '+300', 'children': []}, {'label': 'HbA1c 7.1-8%', 'value': '+350', 'children': []}, {'label': 'HbA1c 8.1-10%', 'value': '拒保', 'children': []}, {'label': 'HbA1c > 10%', 'value': '拒保', 'children': []}]}, {'label': '20-39岁', 'value': '', 'children': [{'label': 'HbA1c < 7%', 'value': '+200', 'children': []}, {'label': 'HbA1c 7.1-8%', 'value': '+250', 'children': []}, {'label': 'HbA1c 8.1-1

In [1]:
historys_list_copy = [{'role': 'user', 'content': '你是一个保险公司的专业核保老师，根据提供的病人基本信息回答问题，病人基本信息:年龄：71岁                临床诊断化验项：_GLU（空腹血糖）:8.82_ALP（碱性磷酸酶）:63.4_GGT（谷氨酰转肽酶）:32.2_TBIL（总胆红素）:5.84_DBIL（直接胆红素）:1.96_AST（谷草转氨酶）:19.70_ALT（谷丙转氨酶）:18.30_TG（甘油三脂）:1.47_BUN（血尿素氮）:5.86_UA（尿酸）:307.50_CR(肌酐):43.60_糖化血红蛋白:9.12_HBSAG（乙肝表面抗原）:阴性(-)_HDL（高密度脂蛋白）:1.24_LDL（低密度脂蛋白）:2.29_糖尿病'}, {'role': 'user', 'content': "\n    请判断该病人是否有如下的情况，情况列表：['病程小于6个月', '病程为6个月－1年', '病程为1年－15年', '病程>15年'],\n    注意返回结果只能是情况列表中的一种，并给出理由，但不要给其他建议说明\n    注意返回结果时：把从列表中选择出的值放到最后，用#和前面的分析理由分隔\n    结果示例：分析理由#选项值\n    "}, {'role': 'assistant', 'content': '根据提供的病人基本信息，该病人年龄为71岁，且临床诊断中明确标示有糖尿病。此外，其糖化血红蛋白（HbA1c）值为9.12%，这个数值显著高于正常范围（通常认为正常值在4%-5.6%之间，但一般不超过6.5%即可诊断为糖尿病），提示患者血糖控制不佳，且其空腹血糖（GLU）为8.82 mmol/L，也高于正常范围（通常在3.9-6.1 mmol/L），进一步证实了其糖尿病状态。虽然没有直接提供糖尿病的病程信息，但基于上述指标，可以合理推测其糖尿病病程可能较长，血糖控制较差。在没有其他信息表明病程较短的情况下，推测其病程超过15年的可能性较大。\n\n分析理由#病程>15年'}, {'role': 'user', 'content': "\n    请判断该病人是否有如下的情况，情况列表：['15-19岁', '20-39岁', '40-49岁', '≥50岁'],\n    注意返回结果只能是情况列表中的一种，并给出理由，但不要给其他建议说明\n    注意返回结果时：把从列表中选择出的值放到最后，用#和前面的分析理由分隔\n    结果示例：分析理由#选项值\n    "}, {'role': 'assistant', 'content': '根据提供的病人基本信息，该病人的年龄为71岁。根据年龄段的划分标准，71岁属于≥50岁的范畴，因此可以直接得出结论。\n\n分析理由#≥50岁'}, {'role': 'user', 'content': "\n    请判断该病人是否有如下的情况，情况列表：['HbA1c < 7%', 'HbA1c 7.1-8%', 'HbA1c 8.1-10%', 'HbA1c > 10%'],\n    注意返回结果只能是情况列表中的一种，并给出理由，但不要给其他建议说明\n    注意返回结果时：把从列表中选择出的值放到最后，用#和前面的分析理由分隔\n    结果示例：分析理由#选项值\n    "}, {'role': 'assistant', 'content': '根据提供的病人基本信息，该病人的糖化血红蛋白（HbA1c）值为9.12%。根据HbA1c的分类标准，9.12%落在8.1-10%的区间内，表明患者的血糖控制较差。\n\n分析理由#HbA1c 8.1-10%'}]
for one_his in historys_list_copy:
    print(one_his)

{'role': 'user', 'content': '你是一个保险公司的专业核保老师，根据提供的病人基本信息回答问题，病人基本信息:年龄：71岁                临床诊断化验项：_GLU（空腹血糖）:8.82_ALP（碱性磷酸酶）:63.4_GGT（谷氨酰转肽酶）:32.2_TBIL（总胆红素）:5.84_DBIL（直接胆红素）:1.96_AST（谷草转氨酶）:19.70_ALT（谷丙转氨酶）:18.30_TG（甘油三脂）:1.47_BUN（血尿素氮）:5.86_UA（尿酸）:307.50_CR(肌酐):43.60_糖化血红蛋白:9.12_HBSAG（乙肝表面抗原）:阴性(-)_HDL（高密度脂蛋白）:1.24_LDL（低密度脂蛋白）:2.29_糖尿病'}
{'role': 'user', 'content': "\n    请判断该病人是否有如下的情况，情况列表：['病程小于6个月', '病程为6个月－1年', '病程为1年－15年', '病程>15年'],\n    注意返回结果只能是情况列表中的一种，并给出理由，但不要给其他建议说明\n    注意返回结果时：把从列表中选择出的值放到最后，用#和前面的分析理由分隔\n    结果示例：分析理由#选项值\n    "}
{'role': 'assistant', 'content': '根据提供的病人基本信息，该病人年龄为71岁，且临床诊断中明确标示有糖尿病。此外，其糖化血红蛋白（HbA1c）值为9.12%，这个数值显著高于正常范围（通常认为正常值在4%-5.6%之间，但一般不超过6.5%即可诊断为糖尿病），提示患者血糖控制不佳，且其空腹血糖（GLU）为8.82 mmol/L，也高于正常范围（通常在3.9-6.1 mmol/L），进一步证实了其糖尿病状态。虽然没有直接提供糖尿病的病程信息，但基于上述指标，可以合理推测其糖尿病病程可能较长，血糖控制较差。在没有其他信息表明病程较短的情况下，推测其病程超过15年的可能性较大。\n\n分析理由#病程>15年'}
{'role': 'user', 'content': "\n    请判断该病人是否有如下的情况，情况列表：['15-19岁', '20-39岁', '40-49岁', '≥50岁'],\n    注意返回结果只能是情况列表中的一种，并给出理

# 第二步，根据“存在并发症和/或其他风险因素”Tree进行多轮问答

In [25]:
#首先判断是否需要进行第二步的多轮问答,
#   诺第一轮最总结论为数值，则进行第二步的多轮问答
#   诺第一轮最总结论为非数值，则不进行第二步的多轮问答
print(end_result)
if not is_number(end_result):
    print("所有问答结束")
else:
    print("进入第二步的多轮问答")

+175
进入第二步的多轮问答


In [26]:
if not is_number(end_result):
    print("所有问答结束")
else:
    # input_data = "" #case 疾病信息
    # historys_list = []
    # message = {"role": "user", "content": f"你是一个保险公司的专业核保老师，根据提供的病人基本信息回答问题，病人基本信息:{input_data}"}
    # historys_list.append(message)
    life_data_map = {"存在并发症和/或其他风险因素":[concl for concl in life_data["Life"] if concl["label"]=="存在并发症和/或其他风险因素"]}
    key = "存在并发症和/或其他风险因素"
    life_data_list = life_data_map[key][0]["children"]
    print(life_data_list)
    question_prompt = """
        请判断该病人是否有如下的情况，情况列表：{node_questions},
        注意返回结果只能是情况列表中的一种，并给出理由，但不要给其他建议说明
        注意返回结果时：把从列表中选择出的值放到最后，用#和前面的分析理由分隔
        结果示例：分析理由#选项值
        """

    run_flag = True
    level_num = 0
    while run_flag:
        print("level num:",level_num)
        node_questions = []
        question_map_level = {}
        for conclusion in life_data_list:
            name_1 = conclusion["label"]
            question_map_level[name_1] = conclusion
            node_questions.append(name_1)
        print(node_questions)
        question_prompt_input = question_prompt.format(node_questions=node_questions)
        print(question_prompt_input)
        message = {"role": "user", "content": question_prompt_input}
        historys_list.append(message)
        messages = historys_list
        answer = qa_base(messages)
        historys_list.append({"role": "assistant", "content": answer})
        print("answer:",answer)

        # # answer = "病程小于6个月"
        # if level_num==0:
        #     answer = '高血压'
        #     historys_list.append({"role": "assistant", "content": answer})
        # if level_num == 1:
        #     answer = "血压评点值"
        #     historys_list.append({"role": "assistant", "content": answer})
        # if level_num == 2:
        #     answer = "如果血压评点小于+100，则同<<高血压>>评点"
        #     historys_list.append({"role": "assistant", "content": answer})
        # level_num += 1
        # #

        answer_key = answer.split("#")[-1]
        #判断是否能得出最终结论
        other_answer = {"children":[],"value":f"answer值：{answer_key},该值未在给定选项中"}
        next_level_qa = question_map_level.get(answer_key,other_answer)
        children_value = next_level_qa["children"]
        #children_value==[] 表示到达了叶子结点，否则继续走分支
        if children_value==[]:
            end_result = next_level_qa["value"]
            print("最终结论",end_result)
            break
        else:
            life_data_list = children_value
            print(children_value)

        
print(f"第一步结论：{first_end_result}")
print(f"第二步结论：{end_result}")
print("historys_list:",historys_list)



[{'label': '高血压', 'value': '', 'children': [{'label': '血压评点值', 'value': '', 'children': [{'label': '如果血压评点小于+100，则同<<高血压>>评点', 'value': '累加按<<高血压>>进行评点', 'children': []}, {'label': '如果血压评点≥+100，则同<<高血压>>评点', 'value': '咨询首席核保师；通常拒保', 'children': []}]}]}, {'label': '存在心脑血管疾病（如冠心病，脑血管病，周围血管疾病）', 'value': '拒保', 'children': []}, {'label': '昏迷病史', 'value': '', 'children': [{'label': '在诊断糖尿病前，发生的糖尿病昏迷病史', 'value': '+0', 'children': []}, {'label': '否则', 'value': '', 'children': [{'label': '昏迷发生时间距今小于1年', 'value': '延期', 'children': []}, {'label': '1 至 2 年', 'value': '再累加+25', 'children': []}, {'label': '> 2年', 'value': '+0', 'children': []}]}]}, {'label': '家族史', 'value': '', 'children': [{'label': '罹患糖尿病的亲属中，同时存在心血管疾病或者肾脏疾病病史', 'value': '+0', 'children': []}, {'label': '其他不利的家族史', 'value': '累加评点：具体见家族史的评点', 'children': []}]}, {'label': '高血脂', 'value': '累加按高胆固醇血症、高甘油三酯血症进行的相应评点', 'children': []}, {'label': '存在糖尿病肾病，或者存在肾功能受损的证据', 'value': '咨询首席核保师；通常拒保', 'children': []}, {'label': '存在糖尿病神经病变', '

In [29]:
for one_history in historys_list:
    print(one_history["role"],":",one_history["content"])

print(f"第一步结论：{first_end_result}")
print(f"第二步结论：{end_result}")

user : 你是一个保险公司的专业核保老师，根据提供的病人基本信息回答问题，病人基本信息:年龄：71岁                临床诊断化验项：_GLU（空腹血糖）:8.82_ALP（碱性磷酸酶）:63.4_GGT（谷氨酰转肽酶）:32.2_TBIL（总胆红素）:5.84_DBIL（直接胆红素）:1.96_AST（谷草转氨酶）:19.70_ALT（谷丙转氨酶）:18.30_TG（甘油三脂）:1.47_BUN（血尿素氮）:5.86_UA（尿酸）:307.50_CR(肌酐):43.60_糖化血红蛋白:9.12_HBSAG（乙肝表面抗原）:阴性(-)_HDL（高密度脂蛋白）:1.24_LDL（低密度脂蛋白）:2.29_糖尿病
user : 
    请判断该病人是否有如下的情况，情况列表：['病程小于6个月', '病程为6个月－1年', '病程为1年－15年', '病程>15年'],
    注意返回结果只能是情况列表中的一种，并给出理由，但不要给其他建议说明
    注意返回结果时：把从列表中选择出的值放到最后，用#和前面的分析理由分隔
    结果示例：分析理由#选项值
    
assistant : 根据提供的病人基本信息，该病人年龄为71岁，且临床诊断中明确标示有糖尿病。此外，其糖化血红蛋白（HbA1c）值为9.12%，这个数值显著高于正常范围（通常认为正常值在4%-5.6%之间，但一般不超过6.5%即可诊断为糖尿病），提示患者血糖控制不佳，且其空腹血糖（GLU）为8.82 mmol/L，也高于正常范围（通常在3.9-6.1 mmol/L），进一步证实了其糖尿病状态。虽然没有直接提供糖尿病的病程信息，但基于上述指标，可以合理推测其糖尿病病程可能较长，血糖控制较差。在没有其他信息表明病程较短的情况下，推测其病程超过15年的可能性较大。

分析理由#病程>15年
user : 
    请判断该病人是否有如下的情况，情况列表：['15-19岁', '20-39岁', '40-49岁', '≥50岁'],
    注意返回结果只能是情况列表中的一种，并给出理由，但不要给其他建议说明
    注意返回结果时：把从列表中选择出的值放到最后，用#和前面的分析理由分隔
    结果示例：分析理由#选项值
    
assistant : 根据提供的病人基本信息，该病人

In [6]:
out_value = ("2",) + ("4",)
print(out_value)

('2', '4')
